In [1]:
# import the modules
import os
from os import listdir
path = r"E:\Photos - imagescanning\Filtered for album\Rishtedaar"
# get the path/directory
def list_files_recursive(path, li):
    
    for entry in os.listdir(path):
            full_path = os.path.join(path, entry)
            if os.path.isdir(full_path):
                list_files_recursive(full_path, li)
            else:
                li.append(full_path)
    return li

list_photos = list_files_recursive(path, [])
# print(list_photos)
print(len(list_photos))

1128


In [2]:
# now we need to scan 2264 photos

import cv2
alg = r"E:\Personal projects\Photos scanning\haarcascade_frontalface_default.xml"
#passing the algorithm to open cv
haar_cascade = cv2.CascadeClassifier(alg)

In [3]:
# trying fro 1 photo first

file_name = list_photos[0]
file_name

'E:\\Photos - imagescanning\\Filtered for album\\Rishtedaar\\A64A4190@8744769.JPG'

In [4]:
# reading the image
img = cv2.imread(file_name, 0)

#creating black and white version of the image
gray_img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

In [5]:
#detecting the faces

faces = haar_cascade.detectMultiScale(gray_img, scaleFactor = 1.05, minNeighbors = 20, minSize = (100,100))

In [6]:
# i = 0

# #for each face detected

# for x, y, w, h in faces:
#     #crop the image to selct only the face
#     cropped_image = img[y : y + h, x : x + w]
#     #loading the target image path to target_file_variable - replace 
#     target_file_name = r"E:\Personal projects\Photos scanning\stored_faces/" + str(i) + '.jpg'
#     cv2.imwrite(target_file_name, cropped_image, )

#     i = i + 1;
    

In [7]:
def scan_images(li_p):
    i = 0
    for im in li_p:
        # print (im)
        file_name = im
        
        img = cv2.imread(file_name, 0)
        
        #creating black and white version of the image
        gray_img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        faces = haar_cascade.detectMultiScale(gray_img, scaleFactor = 1.05, minNeighbors = 20, minSize = (100,100))
        photo_name = os.path.basename(file_name)
        
        #for each face detected
        
        for x, y, w, h in faces:
            #crop the image to selct only the face
            cropped_image = img[y : y + h, x : x + w]
            #loading the target image path to target_file_variable - replace 
            target_file_name = r"E:\Personal projects\Photos scanning\stored_faces/" + photo_name + str(i) + '.jpg'
            # print(target_file_name)
            cv2.imwrite(target_file_name, cropped_image, )
        
            i = i + 1;
    

In [8]:
scan_images(list_photos)

In [9]:
# !pip install face_recognition
import face_recognition

E:\Personal projects\Photos scanning\venv\Lib\site-packages\face_recognition_models\__init__.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


In [10]:
# known_image = face_recognition.load_image_file(r"E:\Personal projects\Photos scanning\stored_faces\0.jpg")
# unknown_image = face_recognition.load_image_file(r"E:\Personal projects\Photos scanning\stored_faces\A64A4203@10238187.JPG0.jpg")

# person_1_encoding = face_recognition.face_encodings(known_image)[0]
# unknown_encoding = face_recognition.face_encodings(unknown_image)[0]

# results = face_recognition.compare_faces([person_1_encoding], unknown_encoding)

In [11]:
# results
import shutil

In [12]:
import os
import shutil
import face_recognition
import numpy as np
import cv2
from sklearn.cluster import DBSCAN


def cluster_faces(folder):

    encodings = []
    image_paths = []

    print("INPUT FOLDER:", folder)

    # --------------------------------
    # STEP 1: LOAD + ENCODE FACES
    # --------------------------------
    for img in os.listdir(folder):

        path = os.path.join(folder, img)

        # Skip folders
        if os.path.isdir(path):
            continue

        print(f"\nProcessing: {img}")

        try:
            # Load image
            image = face_recognition.load_image_file(path)

            # Resize only huge images
            h, w = image.shape[:2]

            if max(h, w) > 2000:
                image = cv2.resize(
                    image,
                    (0, 0),
                    fx=0.7,
                    fy=0.7
                )

            # Detect faces
            face_locations = face_recognition.face_locations(
                image,
                number_of_times_to_upsample=1,
                model="hog"
            )

            print("Faces found:", len(face_locations))

            if len(face_locations) == 0:
                continue

            # Encode faces
            print("Starting encoding...")

            face_encs = face_recognition.face_encodings(
                image,
                known_face_locations=face_locations
            )
            
            print("Encoding complete")

            # Store encodings
            for enc in face_encs:
                encodings.append(enc)
                image_paths.append(path)

        except Exception as e:
            print("ERROR:", e)

    # --------------------------------
    # NO FACES FOUND
    # --------------------------------
    if len(encodings) == 0:
        print("\nNo faces found.")
        return

    # print("\nTotal face encodings:", len(encodings))

    encodings = np.array(encodings)

    # --------------------------------
    # STEP 2: CLUSTER
    # --------------------------------
    clustering = DBSCAN(
        eps=0.5,
        min_samples=1,   # IMPORTANT
        metric="euclidean"
    ).fit(encodings)

    labels = clustering.labels_

    # print("Unique labels:", set(labels))

    # --------------------------------
    # STEP 3: CREATE OUTPUT FOLDERS
    # --------------------------------
    output_base = os.path.join(folder, "clusters")

    # print("\nCreating output folder:")
    # print(output_base)

    os.makedirs(output_base, exist_ok=True)

    # --------------------------------
    # STEP 4: COPY IMAGES
    # --------------------------------
    copied = set()

    for label, img_path in zip(labels, image_paths):

        if label == -1:
            target_folder = os.path.join(output_base, "Unknown")
        else:
            target_folder = os.path.join(output_base, f"Person_{label}")

        os.makedirs(target_folder, exist_ok=True)

        filename = os.path.basename(img_path)

        destination = os.path.join(target_folder, filename)

        # Prevent duplicate copies
        unique_key = (img_path, label)

        if unique_key in copied:
            continue

        copied.add(unique_key)

        try:
            shutil.copy2(img_path, destination)
            # print(f"Copied -> {destination}")

        except Exception as e:
            print("COPY ERROR:", e)

    print("\nClustering complete.")




In [13]:
cluster_faces(r"E:\Personal projects\Photos scanning\stored_faces")

INPUT FOLDER: E:\Personal projects\Photos scanning\stored_faces

Processing: A64A4190@8744769.JPG0.jpg
Faces found: 1
Starting encoding...
Encoding complete

Processing: A64A4190@8744769.JPG1.jpg
Faces found: 1
Starting encoding...
Encoding complete

Processing: A64A4190@8744769.JPG2.jpg
Faces found: 1
Starting encoding...
Encoding complete

Processing: A64A4190@8744769.JPG3.jpg
Faces found: 0

Processing: IMG_6135@13693380.JPG4.jpg
Faces found: 1
Starting encoding...
Encoding complete

Processing: IMG_6135@13693380.JPG5.jpg
Faces found: 1
Starting encoding...
Encoding complete

Processing: IMG_6135@13693380.JPG6.jpg
Faces found: 1
Starting encoding...
Encoding complete

Processing: IMG_6135@13693380.JPG7.jpg
Faces found: 1
Starting encoding...
Encoding complete

Processing: IMG_6147@17035312.JPG10.jpg
Faces found: 1
Starting encoding...
Encoding complete

Processing: IMG_6147@17035312.JPG11.jpg
Faces found: 1
Starting encoding...
Encoding complete

Processing: IMG_6147@17035312.JPG12.

In [14]:
def folder_content_count(root_path):
    result = {}

    for folder_name in os.listdir(root_path):
        folder_path = os.path.join(root_path, folder_name)

        if os.path.isdir(folder_path):
            result[folder_path] = len(os.listdir(folder_path))

    return result

In [15]:
folder_dict = folder_content_count("E:\Personal projects\Photos scanning\stored_faces\clusters")
folder_dict

<>:1: SyntaxWarning: "\P" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\P"? A raw string is also an option.
<>:1: SyntaxWarning: "\P" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\P"? A raw string is also an option.
C:\Users\deepa\AppData\Local\Temp\ipykernel_12132\1831738793.py:1: SyntaxWarning: "\P" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\P"? A raw string is also an option.
  folder_dict = folder_content_count("E:\Personal projects\Photos scanning\stored_faces\clusters")


{'E:\\Personal projects\\Photos scanning\\stored_faces\\clusters\\Person_0': 2489,
 'E:\\Personal projects\\Photos scanning\\stored_faces\\clusters\\Person_1': 28,
 'E:\\Personal projects\\Photos scanning\\stored_faces\\clusters\\Person_10': 2,
 'E:\\Personal projects\\Photos scanning\\stored_faces\\clusters\\Person_11': 4,
 'E:\\Personal projects\\Photos scanning\\stored_faces\\clusters\\Person_12': 3,
 'E:\\Personal projects\\Photos scanning\\stored_faces\\clusters\\Person_13': 1,
 'E:\\Personal projects\\Photos scanning\\stored_faces\\clusters\\Person_14': 1,
 'E:\\Personal projects\\Photos scanning\\stored_faces\\clusters\\Person_15': 1,
 'E:\\Personal projects\\Photos scanning\\stored_faces\\clusters\\Person_16': 4,
 'E:\\Personal projects\\Photos scanning\\stored_faces\\clusters\\Person_17': 1,
 'E:\\Personal projects\\Photos scanning\\stored_faces\\clusters\\Person_18': 1,
 'E:\\Personal projects\\Photos scanning\\stored_faces\\clusters\\Person_19': 4,
 'E:\\Personal projects\\P

In [16]:
import cv2
import os

def rate_image_clarity(folder_path):
    clarity_scores = {}

    for file_name in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file_name)

        # only process images
        if not os.path.isfile(file_path):
            continue

        img = cv2.imread(file_path)

        if img is None:
            continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # clarity score (higher = sharper)
        score = cv2.Laplacian(gray, cv2.CV_64F).var()

        clarity_scores[file_name] = score

    return clarity_scores

In [17]:
clarity_dict = rate_image_clarity(r"E:\Photos - imagescanning\Filtered for album\Rishtedaar")
clarity_dict

{'A64A4190@8744769.JPG': np.float64(354.7902417678004),
 'IMG_6135@13693380.JPG': np.float64(441.621167997236),
 'IMG_6147@17035312.JPG': np.float64(1538.1369832985386),
 'IMG_6150@11970578.JPG': np.float64(314.9891467326258),
 'IMG_6157@15577700.JPG': np.float64(1321.3630337395755),
 'IMG_6160@15386600.JPG': np.float64(1381.4033342648145),
 'IMG_6186@15140355.JPG': np.float64(1064.3909339210336),
 'IMG_6200@14425995.JPG': np.float64(460.69381253926827),
 'IMG_6205@13776932.JPG': np.float64(374.66800142309273),
 'IMG_6210@14040495.JPG': np.float64(610.9623840972504),
 'IMG_6234@15066589.JPG': np.float64(1160.7492158362957),
 'IMG_6239@13737730.JPG': np.float64(726.6513773010468),
 'IMG_6241@12480379.JPG': np.float64(360.97969454312727),
 'IMG_6261@14560917.JPG': np.float64(681.9239322346049),
 'IMG_6263@13071450.JPG': np.float64(429.04994725935865),
 'IMG_6268@14622831.JPG': np.float64(1459.167397187683),
 'IMG_6275@11727754.JPG': np.float64(370.5037701665712),
 'IMG_6278@16563954.JPG'

In [18]:
import os
import shutil
import re


# =====================================================
# SAFE NORMALIZER (FIXES YOUR "JPG137.JPG" ISSUE)
# =====================================================
def normalize(img_name):
    """
    Converts:
    IMG_6539@15929083.JPG137.jpg → IMG_6539@15929083.JPG
    IMG_6539@15929083.JPG5.jpg   → IMG_6539@15929083.JPG
    """

    name = img_name.lower()

    # remove trailing noise like 137.jpg, 5.jpg, etc.
    name = re.split(r'\.jpg\d*', name)[0]

    # restore canonical .JPG format
    if ".jpg" in name:
        base = name.split(".jpg")[0]
        return base + ".JPG"

    return name + ".JPG"


# =====================================================
# MAIN PIPELINE
# =====================================================
def process_folders(folders_dict, source_dir, clarity_dict, output_dir):

    # -----------------------------
    # TYPE SAFETY
    # -----------------------------
    if not isinstance(folders_dict, dict):
        raise ValueError(f"folders_dict must be dict, got {type(folders_dict)}")

    if not isinstance(clarity_dict, dict):
        raise ValueError(f"clarity_dict must be dict, got {type(clarity_dict)}")

    os.makedirs(output_dir, exist_ok=True)

    # sort folders by size (smallest first)
    sorted_folders = sorted(folders_dict.items(), key=lambda x: x[1])

    remaining = dict(folders_dict)

    # normalize clarity dict once
    clarity_norm = {k.upper(): v for k, v in clarity_dict.items()}


    for folder_path, count in sorted_folders:

        if folder_path not in remaining:
            continue

        if not os.path.isdir(folder_path):
            print("Skipping missing folder:", folder_path)
            continue

        images = os.listdir(folder_path)

        if len(images) == 0:
            continue

        print("\nProcessing:", folder_path, "| Images:", len(images))


        # =====================================================
        # CASE 1: SINGLE IMAGE (DIRECT RESOLUTION)
        # =====================================================
        if len(images) == 1:

            img = images[0]
            clean = normalize(img).upper()

            src_path = os.path.join(source_dir, clean)
            dst_path = os.path.join(output_dir, clean)

            print("SRC:", src_path)

            if os.path.exists(src_path):
                shutil.copy(src_path, dst_path)
                print("Copied (single):", clean)
            else:
                print("Missing source:", src_path)


        # =====================================================
        # CASE 2: MULTIPLE IMAGES (CLARITY SELECTION)
        # =====================================================
        else:

            best_img = None
            best_clean = None
            best_score = -1

            for img in images:

                clean = normalize(img).upper()
                score = clarity_norm.get(clean, -1)

                if score > best_score:
                    best_score = score
                    best_img = img
                    best_clean = clean

            if best_clean is None:
                continue

            src_path = os.path.join(source_dir, best_clean)
            dst_path = os.path.join(output_dir, best_clean)

            print("SRC:", src_path)

            if os.path.exists(src_path):
                shutil.copy(src_path, dst_path)
                print("Copied (multi):", best_clean)
            else:
                print("Missing source:", src_path)


        # =====================================================
        # GLOBAL PROPAGATION RULE (PREFIX-BASED REMOVAL)
        # =====================================================
        saved = normalize(images[0]).upper() if len(images) == 1 else best_clean

        to_remove = []

        for f in list(remaining.keys()):

            if not os.path.isdir(f):
                continue

            try:
                f_images = os.listdir(f)
            except:
                continue

            for im in f_images:

                im_clean = normalize(im).upper()

                if im_clean.startswith(saved):
                    to_remove.append(f)
                    break

        for f in to_remove:
            del remaining[f]
            print("Removed folder:", f)

In [19]:
process_folders(folder_dict,  r"E:\Photos - imagescanning\Filtered for album\Rishtedaar", clarity_dict, r"E:\Photos - imagescanning\Selected")


Processing: E:\Personal projects\Photos scanning\stored_faces\clusters\Person_13 | Images: 1
SRC: E:\Photos - imagescanning\Filtered for album\Rishtedaar\IMG_8395@11046737.JPG
Copied (single): IMG_8395@11046737.JPG
Removed folder: E:\Personal projects\Photos scanning\stored_faces\clusters\Person_13

Processing: E:\Personal projects\Photos scanning\stored_faces\clusters\Person_14 | Images: 1
SRC: E:\Photos - imagescanning\Filtered for album\Rishtedaar\IMG_8442@10494770.JPG
Copied (single): IMG_8442@10494770.JPG
Removed folder: E:\Personal projects\Photos scanning\stored_faces\clusters\Person_0
Removed folder: E:\Personal projects\Photos scanning\stored_faces\clusters\Person_14

Processing: E:\Personal projects\Photos scanning\stored_faces\clusters\Person_15 | Images: 1
SRC: E:\Photos - imagescanning\Filtered for album\Rishtedaar\L50A4087@7768388.JPG
Copied (single): L50A4087@7768388.JPG
Removed folder: E:\Personal projects\Photos scanning\stored_faces\clusters\Person_15

Processing: E: